In [ ]:
import sys; sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils
import periodic_unit_helper
import numpy.linalg as la

In [ ]:
parallelism.set_max_num_tbb_threads(8)

In [ ]:
import importlib

In [ ]:
importlib.reload(parametric_pillows)

In [ ]:
triArea = 0.0002

In [ ]:
from ipywidgets import interactive, widgets
def plotForAlpha(alpha): visualization.plot_line_segments(*parametric_pillows.logSpiralPlot(alpha=alpha, edgeLength=0.02, minDist=0.1, margin=.01))
iplot = interactive(plotForAlpha, alpha = widgets.FloatSlider(min=1, max=90, value=70, step=1))
iplot.children[-1].layout.height = '500px'
display(iplot)

In [ ]:
pts, edges = parametric_pillows.logSpiralPlot(alpha=70, edgeLength=0.02, minDist=0.1, margin=.01)

In [ ]:
m, fuseMarkers, fuseEdges = wall_generation.triangulate_channel_walls(pts, edges, triArea)
visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=5, height=5)

In [ ]:
def get_average_edge_length(pts, fuseSegments):
    total_edge_length = 0
    for e in fuseSegments:
        total_edge_length += la.norm(pts[e[0]] - pts[e[1]])
    return total_edge_length / len(fuseSegments)

In [ ]:
avg_len = get_average_edge_length(m.vertices()[:, :2], fuseEdges)

In [ ]:
avg_len

In [ ]:
import igl

In [ ]:
def get_scaled_box(pts, scale = 1.1):
    cm = np.mean(np.array(pts), axis = 0)
    scaled_pts = (np.array(pts) - cm) * scale + cm
    return igl.bounding_box(scaled_pts)

In [ ]:
def get_scaled_square_box(pts, scale = 1.1):
    bbox_vx, bbox_edges = get_scaled_box(pts, scale)
    box_width = np.abs(bbox_vx[0][0] - bbox_vx[2][0])
    box_height = np.abs(bbox_vx[0][1] - bbox_vx[1][1])
    if box_width > box_height:
        height_scale = box_width / box_height
        height_mean = np.mean(bbox_vx, axis = 0)[1]
        bbox_vx[:, 1] = (bbox_vx[:, 1] - height_mean) * height_scale + height_mean
    else:
        width_scale = box_height / box_width
        width_mean = np.mean(bbox_vx, axis = 0)[0]
        bbox_vx[:, 0] = (bbox_vx[:, 0] - width_mean) * width_scale + width_mean
    return bbox_vx, bbox_edges

In [ ]:
def get_box_with_dimension(pts, width, height):
    bbox_vx, bbox_edges = get_scaled_box(pts, 1)
    box_width = np.abs(bbox_vx[0][0] - bbox_vx[2][0])
    box_height = np.abs(bbox_vx[0][1] - bbox_vx[1][1])

    height_scale = height / box_height
    height_mean = np.mean(bbox_vx, axis = 0)[1]
    bbox_vx[:, 1] = (bbox_vx[:, 1] - height_mean) * height_scale + height_mean

    width_scale = width / box_width
    width_mean = np.mean(bbox_vx, axis = 0)[0]
    bbox_vx[:, 0] = (bbox_vx[:, 0] - width_mean) * width_scale + width_mean
    return bbox_vx, bbox_edges

In [ ]:
bbox_vx, bbox_edges = get_scaled_box(pts, scale = 1.1)

In [ ]:
bbox_vx, bbox_edges = get_scaled_square_box(pts, scale = 1.1)

In [ ]:
box_width = np.abs(bbox_vx[0][0] - bbox_vx[2][0])
box_height = np.abs(bbox_vx[0][1] - bbox_vx[1][1])

num_width_seg = int(np.round(box_width / avg_len))
num_height_seg = int(np.round(box_height / avg_len))

top_y = bbox_vx[0][1]
right_x = bbox_vx[0][0]
bot_y = bbox_vx[3][1]
left_x = bbox_vx[3][0]

In [ ]:
top_y,right_x, bot_y, left_x

In [ ]:
num_width_seg, num_height_seg

In [ ]:
bbox_vx = np.concatenate((np.linspace(bbox_vx[0], bbox_vx[1], num_height_seg),  np.linspace(bbox_vx[1], bbox_vx[3], num_width_seg)[1:], np.linspace(bbox_vx[3], bbox_vx[2], num_height_seg)[1:], np.linspace(bbox_vx[2], bbox_vx[0], num_width_seg)[1:-1]))

In [ ]:
bbox_edges = [[i, i + 1] for i in np.arange(len(bbox_vx) - 1)] + [[len(bbox_vx) - 1, 0]]

In [ ]:
n_vx = pts + list(bbox_vx)
n_edge = edges + list(np.array(bbox_edges) + len(pts))

In [ ]:
visualization.plot_line_segments(n_vx, n_edge)

In [ ]:
import importlib

In [ ]:
importlib.reload(parametric_pillows)

In [ ]:
from parametric_pillows import get_perioidic_mesh

In [ ]:
# n_vx

In [ ]:
m, fuseMarkers, fuseSegments = wall_generation.triangulate_channel_walls(n_vx, n_edge, triArea, flags="Y")


In [ ]:
def is_bbox(point):
    if (point[0] == left_x or point[0] == right_x or point[1] == top_y or point[1] == bot_y):
        return True
    return False

In [ ]:
visualization.plot_2d_mesh(m, pointList=np.where(np.logical_and(np.array(fuseMarkers) == 1, [(not is_bbox(pt)) for pt in m.vertices()]))[0], width=5, height=5)

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, np.logical_and(np.array(fuseMarkers) == 1, [(not is_bbox(pt)) for pt in m.vertices()]), epsilon = 1e-5)

In [ ]:
import periodic_unit_helper

In [ ]:
fixedVars = periodic_unit_helper.get_center_fixedVars(ipu)

In [ ]:
# isheet.setRelaxedStiffnessEpsilon(1e-6)

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
viewer.showWireframe(True)

In [ ]:
ipu.sheet.rigidMotionPinVars

In [ ]:
fd_perturb = np.random.uniform(-1e-3, 1e-3, ipu.numVars())

In [ ]:
ipu.setVars(ipu.getVars() + fd_perturb)

In [ ]:
ipu.sheet.volume()

In [ ]:
ipu.periodicVolume()

In [ ]:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])



In [ ]:
import time, vis
benchmark.reset()
ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.pressure = 17
opts.niter = 200
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb)
benchmark.report()

In [ ]:
viewer.update()


In [ ]:
ipu.getVars()[:3]

### Vibrational Mode analysis


In [ ]:
import compute_vibrational_modes

In [ ]:
class ModalAnalysisWrapper:
    def __init__(self, sheet):
        self.sheet = sheet
    def hessian(self):
        return self.sheet.hessian(inflation.InflatableSheet.EnergyType.Elastic)

In [ ]:
fixedVars

In [ ]:
lambdas, modes = compute_vibrational_modes.compute_vibrational_modes(ModalAnalysisWrapper(ipu), mtype=compute_vibrational_modes.MassMatrixType.FULL, n=16, sigma=-1e-10, fixedVars = fixedVars)

import mode_viewer, importlib
importlib.reload(mode_viewer);
mview = mode_viewer.ModeViewer(ipu, modes, lambdas, amplitude=10)
# mview.showScalarField(rod_colors)
mview.show()

### Validate periodic volume

In [ ]:
sheet = ipu.sheet

In [ ]:
def get_triangles_of_boundary_edges(edge):
    tri1 = [sheet.getDeformedVtxPosition(edge[0], 0), sheet.getDeformedVtxPosition(edge[1], 0), sheet.getDeformedVtxPosition(edge[1], 1)]
    tri2 = [sheet.getDeformedVtxPosition(edge[0], 0), sheet.getDeformedVtxPosition(edge[1], 1), sheet.getDeformedVtxPosition(edge[0], 1)]
    return [tri1, tri2]

In [ ]:
tris = []
for edge in mesh.boundaryElements():
    tris += get_triangles_of_boundary_edges(edge)

In [ ]:
len(tris)

In [ ]:
volume = 0
for tri in tris:
    volume += la.det(tri)

In [ ]:
volume / 6

### Validate gradient

In [ ]:
import fd_validation

In [ ]:
class fd_wrapper:
    def __init__(self, ipu):
        self.ipu = ipu

    def setVars(self, v):
        self.ipu.sheet.setVars(v)
    def numVars(self):
        return self.ipu.sheet.numVars()

    def getVars(self):
        return self.ipu.sheet.getVars()

    def energy(self):   return self.ipu.energyPeriodicPressurePotential()
    def gradient(self): return self.ipu.gradientPeriodicPressurePotential()

In [ ]:
fd_validation.gradConvergencePlot(fd_wrapper(ipu))